# 1. Context

> **TODO — slot not started yet.**
>
> Add here: project objective, business question(s) this EDA supports, and what decisions the pipeline design will depend on.

# 2. Data Sources

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "vscode"
import os

raw_path = "../data/raw"

os.listdir(raw_path)

['original_cleaned_nyc_taxi_data_2018.csv',
 'taxi_zone_geo.csv',
 '.complete',
 'taxi_trip_data.csv']

In [2]:
trips_full = pd.read_csv("../data/raw/taxi_trip_data.csv")
trips = trips_full.sample(n=100_000, random_state=42)

# Loading the geographics zones .csv
zones = pd.read_csv("../data/raw/taxi_zone_geo.csv")

# Loading "clean" data
cleaned = pd.read_csv("../data/raw/original_cleaned_nyc_taxi_data_2018.csv", nrows=100_000)

# 3. Overview of the 3 datasets

## Quick Inspection

In [3]:
def quick(df):
    print("shape:", df.shape)
    print("\ncolumns:")
    print(df.columns)
    print("\ndtypes:")
    print(df.dtypes)
    print("\nisnull:")
    print(df.isnull().sum())
    display(df.head())

## Trips

In [4]:
quick(trips)

shape: (100000, 17)

columns:
Index(['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count',
       'trip_distance', 'rate_code', 'store_and_fwd_flag', 'payment_type',
       'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'imp_surcharge', 'total_amount', 'pickup_location_id',
       'dropoff_location_id'],
      dtype='object')

dtypes:
vendor_id                int64
pickup_datetime         object
dropoff_datetime        object
passenger_count          int64
trip_distance          float64
rate_code                int64
store_and_fwd_flag      object
payment_type             int64
fare_amount            float64
extra                  float64
mta_tax                float64
tip_amount             float64
tolls_amount           float64
imp_surcharge          float64
total_amount           float64
pickup_location_id       int64
dropoff_location_id      int64
dtype: object

isnull:
vendor_id              0
pickup_datetime        0
dropoff_datetime     

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
919213,2,2018-11-05 12:00:15,2018-11-05 12:37:47,1,5.57,1,N,1,26.0,0.0,0.5,5.36,0.0,0.3,32.16,230,13
9467153,2,2018-10-24 23:02:35,2018-10-24 23:26:49,1,5.42,1,N,1,21.5,0.5,0.5,4.56,0.0,0.3,27.36,148,263
6585777,1,2018-10-11 16:06:15,2018-10-11 16:06:22,1,0.00,1,N,2,2.5,1.0,0.5,0.00,0.0,0.3,4.30,237,237
3878022,1,2018-10-12 17:13:45,2018-10-12 17:59:11,1,7.10,1,N,1,31.5,1.0,0.5,4.99,0.0,0.3,38.29,170,49
5537116,2,2018-03-03 18:02:08,2018-03-03 18:32:46,3,4.77,1,N,2,22.0,0.0,0.5,0.00,0.0,0.3,22.80,249,236


## Zones

In [5]:
quick(zones)

shape: (263, 4)

columns:
Index(['zone_id', 'zone_name', 'borough', 'zone_geom'], dtype='object')

dtypes:
zone_id       int64
zone_name    object
borough      object
zone_geom    object
dtype: object

isnull:
zone_id      0
zone_name    0
borough      0
zone_geom    0
dtype: int64


,zone_id,zone_name,borough,zone_geom
0,1,Newark Airport,EWR,"POLYGON((-74.1856319999999 40.6916479999999, -..."
1,3,Allerton/Pelham Gardens,Bronx,"POLYGON((-73.848596761 40.8716707849999, -73.8..."
2,18,Bedford Park,Bronx,"POLYGON((-73.8844286139999 40.8668003789999, -..."
3,20,Belmont,Bronx,"POLYGON((-73.8839239579998 40.8644177609999, -..."
4,31,Bronx Park,Bronx,"POLYGON((-73.8710017319999 40.8572767429999, -..."


## Cleaned

In [6]:
quick(cleaned)

shape: (100000, 21)

columns:
Index(['Unnamed: 0', 'trip_distance', 'rate_code', 'store_and_fwd_flag',
       'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'imp_surcharge', 'total_amount', 'pickup_location_id',
       'dropoff_location_id', 'year', 'month', 'day', 'day_of_week',
       'hour_of_day', 'trip_duration', 'calculated_total_amount'],
      dtype='object')

dtypes:
Unnamed: 0                   int64
trip_distance              float64
rate_code                    int64
store_and_fwd_flag          object
payment_type                 int64
fare_amount                float64
extra                      float64
mta_tax                    float64
tip_amount                 float64
tolls_amount               float64
imp_surcharge              float64
total_amount               float64
pickup_location_id           int64
dropoff_location_id          int64
year                         int64
month                        int64
day                 

,Unnamed: 0,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,...,total_amount,pickup_location_id,dropoff_location_id,year,month,day,day_of_week,hour_of_day,trip_duration,calculated_total_amount
0,3,16.97,1,N,1,49.5,0.0,0.5,5.61,5.76,...,61.67,231,138,2018,3,29,3,13,3317.0,61.67
1,4,14.45,1,N,1,45.5,0.0,0.5,10.41,5.76,...,62.47,87,138,2018,3,29,3,14,3648.0,62.47
2,5,11.60,1,N,1,42.0,0.0,0.5,14.57,5.76,...,63.13,68,138,2018,3,29,3,14,3540.0,63.13
3,10,5.10,1,N,1,26.5,1.0,0.5,5.65,0.00,...,33.95,186,33,2018,3,29,3,16,2585.0,33.95
4,12,11.11,1,N,1,45.5,1.0,0.5,10.61,5.76,...,63.67,163,138,2018,3,29,3,16,4521.0,63.67


## General view

In [7]:
datasets = {
    "trips": trips,
    "zones": zones,
    "cleaned": cleaned
}

for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nTypes:")
    print(df.dtypes)


===== trips =====
Shape: (100000, 17)
Columns: ['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'rate_code', 'store_and_fwd_flag', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'imp_surcharge', 'total_amount', 'pickup_location_id', 'dropoff_location_id']

Types:
vendor_id                int64
pickup_datetime         object
dropoff_datetime        object
passenger_count          int64
trip_distance          float64
rate_code                int64
store_and_fwd_flag      object
payment_type             int64
fare_amount            float64
extra                  float64
mta_tax                float64
tip_amount             float64
tolls_amount           float64
imp_surcharge          float64
total_amount           float64
pickup_location_id       int64
dropoff_location_id      int64
dtype: object

===== zones =====
Shape: (263, 4)
Columns: ['zone_id', 'zone_name', 'borough', 'zone_geom']

Types:
zone_id       int64

We are choosing the "taxi_trip_data" as our main source of information because is the .csv that only has information about the taxi trips, and that is what we are going to analyse in this project

# 4. Relationship between datasets

> **TODO — slot not started yet.**
>
> Explore how `taxi_trip_data` relates to `taxi_zone_geo` (via `pickup_location_id` / `dropoff_location_id`) and how `cleaned` compares to the raw `taxi_trip_data` (same source, pre-cleaned reference?).

# 5. EDA - taxi_trip_data.csv

## 5.1 Structure and types

In [8]:
# import pandas as pd

# #Loading the taxi trips .csv
# trips = pd.read_csv("../data/raw/taxi_trip_data.csv", nrows=100_000)

In [9]:
trips.shape

(100000, 17)

In [10]:
trips.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
919213,2,2018-11-05 12:00:15,2018-11-05 12:37:47,1,5.57,1,N,1,26.0,0.0,0.5,5.36,0.0,0.3,32.16,230,13
9467153,2,2018-10-24 23:02:35,2018-10-24 23:26:49,1,5.42,1,N,1,21.5,0.5,0.5,4.56,0.0,0.3,27.36,148,263
6585777,1,2018-10-11 16:06:15,2018-10-11 16:06:22,1,0.00,1,N,2,2.5,1.0,0.5,0.00,0.0,0.3,4.30,237,237
3878022,1,2018-10-12 17:13:45,2018-10-12 17:59:11,1,7.10,1,N,1,31.5,1.0,0.5,4.99,0.0,0.3,38.29,170,49
5537116,2,2018-03-03 18:02:08,2018-03-03 18:32:46,3,4.77,1,N,2,22.0,0.0,0.5,0.00,0.0,0.3,22.80,249,236


In [11]:
trips.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 919213 to 2551419
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   vendor_id            100000 non-null  int64  
 1   pickup_datetime      100000 non-null  object 
 2   dropoff_datetime     100000 non-null  object 
 3   passenger_count      100000 non-null  int64  
 4   trip_distance        100000 non-null  float64
 5   rate_code            100000 non-null  int64  
 6   store_and_fwd_flag   100000 non-null  object 
 7   payment_type         100000 non-null  int64  
 8   fare_amount          100000 non-null  float64
 9   extra                100000 non-null  float64
 10  mta_tax              100000 non-null  float64
 11  tip_amount           100000 non-null  float64
 12  tolls_amount         100000 non-null  float64
 13  imp_surcharge        100000 non-null  float64
 14  total_amount         100000 non-null  float64
 15  pickup_location_

"pickup_datetime" and "dropoff_datetime" as object is "wrong" i must change the type for 'date' in the pipeline

payment_type as a number, i found in the description of the dataset the equivalent numbers, so in the future i will need to create a auxiliary table

rate_code is categoric besides beeing int

store_and_fwd_flg is a object but is categoric

In [12]:
schema = pd.DataFrame({
    "column": trips.columns,
    "dtype": trips.dtypes.values,
    "null_count": trips.isnull().sum().values,
    "unique_count": trips.nunique().values
})

schema

,column,dtype,null_count,unique_count
0,vendor_id,int64,0,3
1,pickup_datetime,object,0,99695
2,dropoff_datetime,object,0,99730
3,passenger_count,int64,0,9
4,trip_distance,float64,0,2904
5,rate_code,int64,0,7
6,store_and_fwd_flag,object,0,2
7,payment_type,int64,0,4
8,fare_amount,float64,0,764
9,extra,float64,0,12


17 columns
100,000 records in the sample
0 nulls across all columns in the sample
8 float64
6 int64
3 object

## 5.2 Data quality

In [13]:
trips.isnull().sum()

vendor_id              0
pickup_datetime        0
dropoff_datetime       0
passenger_count        0
trip_distance          0
rate_code              0
store_and_fwd_flag     0
payment_type           0
fare_amount            0
extra                  0
mta_tax                0
tip_amount             0
tolls_amount           0
imp_surcharge          0
total_amount           0
pickup_location_id     0
dropoff_location_id    0
dtype: int64

In [14]:
trips.duplicated().sum()

np.int64(65)

In [15]:
trips['payment_type'].value_counts()

payment_type
1    82683
2    16098
3      958
4      261
Name: count, dtype: int64

In [16]:
trips["vendor_id"].value_counts()

vendor_id
2    60249
1    39291
4      460
Name: count, dtype: int64

In [17]:
trips["rate_code"].value_counts()

rate_code
1     90919
2      4601
5      2405
3      1628
4       432
99       12
6         3
Name: count, dtype: int64

In [18]:
trips[trips["trip_distance"] <= 0].count()

vendor_id              2734
pickup_datetime        2734
dropoff_datetime       2734
passenger_count        2734
trip_distance          2734
rate_code              2734
store_and_fwd_flag     2734
payment_type           2734
fare_amount            2734
extra                  2734
mta_tax                2734
tip_amount             2734
tolls_amount           2734
imp_surcharge          2734
total_amount           2734
pickup_location_id     2734
dropoff_location_id    2734
dtype: int64

In [19]:
trips[trips ['pickup_location_id'] == trips['dropoff_location_id']].count()

vendor_id              6062
pickup_datetime        6062
dropoff_datetime       6062
passenger_count        6062
trip_distance          6062
rate_code              6062
store_and_fwd_flag     6062
payment_type           6062
fare_amount            6062
extra                  6062
mta_tax                6062
tip_amount             6062
tolls_amount           6062
imp_surcharge          6062
total_amount           6062
pickup_location_id     6062
dropoff_location_id    6062
dtype: int64

Now we will analyse this lines to understand whats happening

In [20]:
trips_wrongs = trips[trips["trip_distance"] <= 0]
trips_wrongs.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
6585777,1,2018-10-11 16:06:15,2018-10-11 16:06:22,1,0.0,1,N,2,2.5,1.0,0.5,0.00,0.0,0.3,4.30,237,237
9192524,1,2018-01-30 09:46:12,2018-01-30 10:14:49,1,0.0,1,N,2,2.5,0.0,0.5,0.00,0.0,0.3,3.30,161,263
8681947,2,2018-03-13 01:01:42,2018-03-13 01:02:01,1,0.0,1,N,2,2.5,0.5,0.5,0.00,0.0,0.3,3.80,119,119
789701,1,2018-02-08 23:12:08,2018-02-08 23:12:17,1,0.0,1,N,3,2.5,0.5,0.5,0.00,0.0,0.3,3.80,106,106
8467201,2,2018-05-19 01:43:10,2018-05-19 01:43:12,1,0.0,5,N,1,80.0,0.0,0.5,16.16,0.0,0.3,96.96,264,90


In [21]:
trips_wrongs.shape

(2734, 17)

In [22]:
duplicates = trips[trips.duplicated(keep=False)]
#keep = false, show from the first apparence to the final, if we keep=True, will only show from the second ocorrence

In [23]:
duplicates.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
7004181,2,2018-03-25 10:07:19,2018-03-25 10:08:15,2,0.36,1,N,1,3.0,0.0,0.5,0.76,0.00,0.3,4.56,263,236
8312687,2,2018-03-20 17:37:22,2018-03-20 18:19:43,1,12.42,1,N,1,39.0,1.0,0.5,9.31,5.76,0.3,55.87,138,107
627106,2,2018-03-12 23:50:13,2018-03-13 00:07:50,2,8.86,1,N,1,25.5,0.5,0.5,5.36,0.00,0.3,32.16,138,17
810955,2,2018-03-31 07:54:12,2018-03-31 08:13:25,1,7.79,1,N,1,24.5,0.0,0.5,6.32,0.00,0.3,31.62,137,106
4584243,2,2018-03-06 13:53:12,2018-03-06 14:26:00,1,6.77,1,N,1,27.0,0.0,0.5,6.95,0.00,0.3,34.75,88,142


In [24]:
duplicates.shape

(130, 17)

In [25]:
duplicates.sort_values(
    by=["pickup_datetime", "dropoff_datetime"]
).head(10)

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
7672992,2,2018-03-01 11:59:45,2018-03-01 12:23:45,1,9.54,1,N,1,29.5,0.0,0.5,6.06,0.00,0.3,36.36,138,49
6219724,2,2018-03-01 11:59:45,2018-03-01 12:23:45,1,9.54,1,N,1,29.5,0.0,0.5,6.06,0.00,0.3,36.36,138,49
5170995,2,2018-03-01 15:50:16,2018-03-01 16:34:59,6,8.67,1,N,1,30.0,0.0,0.5,0.00,0.00,0.3,30.80,138,216
3399317,2,2018-03-01 15:50:16,2018-03-01 16:34:59,6,8.67,1,N,1,30.0,0.0,0.5,0.00,0.00,0.3,30.80,138,216
2648634,1,2018-03-01 20:56:37,2018-03-01 21:21:26,1,6.80,1,N,1,23.5,0.5,0.5,4.95,0.00,0.3,29.75,237,198
683939,1,2018-03-01 20:56:37,2018-03-01 21:21:26,1,6.80,1,N,1,23.5,0.5,0.5,4.95,0.00,0.3,29.75,237,198
7305277,2,2018-03-02 05:44:27,2018-03-02 06:13:16,1,17.41,3,N,1,64.0,0.5,0.0,16.21,16.26,0.3,97.27,264,264
8977404,2,2018-03-02 05:44:27,2018-03-02 06:13:16,1,17.41,3,N,1,64.0,0.5,0.0,16.21,16.26,0.3,97.27,264,264
2611122,2,2018-03-02 10:17:49,2018-03-02 10:40:42,1,13.88,1,N,1,37.5,0.0,0.5,11.49,0.00,0.3,49.79,138,203
8016325,2,2018-03-02 10:17:49,2018-03-02 10:40:42,1,13.88,1,N,1,37.5,0.0,0.5,11.49,0.00,0.3,49.79,138,203


I identified 65 duplicated lines, we can see that all columns are duplicates so probably are really duplicates but to have more confidence i will run another lines to analyse especifically other columns

Running another checkup i can confirm that is really duplicates, because we can see that ALL columns are duplicates too

In the 100.000 we found 65 duplicates that translate to 130 lines

So in the pipeline this duplicates will be removed

In [26]:
duplicate_rows = len(duplicates)
percentage = duplicate_rows / len(trips) * 100

print(f"Rows involved in duplication: {duplicate_rows}")
print(f"Percentage: {percentage:.3f}%")

Rows involved in duplication: 130
Percentage: 0.130%


In [27]:
duplicate_counts = trips.value_counts()

duplicate_counts[duplicate_counts > 1].head(20)

vendor_id  pickup_datetime      dropoff_datetime     passenger_count  trip_distance  rate_code  store_and_fwd_flag  payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  imp_surcharge  total_amount  pickup_location_id  dropoff_location_id
2          2018-03-15 15:31:28  2018-03-15 16:15:28  1                4.36           1          N                   1             27.5         0.0    0.5      5.66        0.00          0.3            33.96         113                 236                    2
1          2018-03-19 09:28:56  2018-03-19 10:14:46  1                8.10           1          N                   1             33.0         0.0    0.5      7.91        5.76          0.3            47.47         138                 162                    2
2          2018-03-03 02:02:39  2018-03-03 02:02:41  0                0.00           5          N                   1             119.8        0.0    0.5      24.12       0.00          0.3            144.72        264           

In [28]:
trips_wrongs = trips[trips["trip_distance"] <= 0]
trips_wrongs.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
6585777,1,2018-10-11 16:06:15,2018-10-11 16:06:22,1,0.0,1,N,2,2.5,1.0,0.5,0.00,0.0,0.3,4.30,237,237
9192524,1,2018-01-30 09:46:12,2018-01-30 10:14:49,1,0.0,1,N,2,2.5,0.0,0.5,0.00,0.0,0.3,3.30,161,263
8681947,2,2018-03-13 01:01:42,2018-03-13 01:02:01,1,0.0,1,N,2,2.5,0.5,0.5,0.00,0.0,0.3,3.80,119,119
789701,1,2018-02-08 23:12:08,2018-02-08 23:12:17,1,0.0,1,N,3,2.5,0.5,0.5,0.00,0.0,0.3,3.80,106,106
8467201,2,2018-05-19 01:43:10,2018-05-19 01:43:12,1,0.0,5,N,1,80.0,0.0,0.5,16.16,0.0,0.3,96.96,264,90


In [29]:
trips_wrongs.shape

(2734, 17)

In [30]:
trips_wrongs_rows = len(trips_wrongs)
percentage = trips_wrongs_rows / len(trips) * 100

print(f"Rows involved in trip_distance <= 0: {trips_wrongs_rows}")
print(f"Percentage: {percentage:.3f}%")

Rows involved in trip_distance <= 0: 2734
Percentage: 2.734%


2734 in 100.000, we talking about less than 3%

In [31]:
trips_location_wrong = trips[trips['pickup_location_id'] == trips["dropoff_location_id"]]

trips_location_wrong.shape

(6062, 17)

In [32]:
trips_location_wrong[trips_location_wrong['trip_distance'] <= 0].shape

(1995, 17)

In [33]:
trips_location_wrong[trips_location_wrong['trip_distance'] < 0].shape

(0, 17)

In [34]:
trips_location_wrong.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
6585777,1,2018-10-11 16:06:15,2018-10-11 16:06:22,1,0.0,1,N,2,2.5,1.0,0.5,0.0,0.0,0.3,4.3,237,237
4521825,1,2018-11-03 17:46:23,2018-11-03 19:02:16,1,14.5,1,N,1,55.0,0.0,0.5,13.9,0.0,0.3,69.7,236,236
8681947,2,2018-03-13 01:01:42,2018-03-13 01:02:01,1,0.0,1,N,2,2.5,0.5,0.5,0.0,0.0,0.3,3.8,119,119
789701,1,2018-02-08 23:12:08,2018-02-08 23:12:17,1,0.0,1,N,3,2.5,0.5,0.5,0.0,0.0,0.3,3.8,106,106
959408,1,2018-01-30 13:05:20,2018-01-30 13:06:38,1,0.2,1,N,2,3.0,0.0,0.5,0.0,0.0,0.3,3.8,236,236


In [35]:
trips_no_distance = trips_location_wrong[trips_location_wrong['trip_distance'] == 0]
trips_no_distance.shape

(1995, 17)

In [36]:
trips_no_distance[trips_no_distance["fare_amount"] == 0].shape

(35, 17)

In [37]:
trips_no_distance[trips_no_distance["fare_amount"] < 0].shape

(39, 17)

In [38]:
fare_invalid = trips_no_distance[
    trips_no_distance["fare_amount"] <= 0
]

fare_invalid[[
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "payment_type",
    "rate_code"
]]

,pickup_datetime,dropoff_datetime,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount,payment_type,rate_code
5493132,2018-12-29 05:22:29,2018-12-29 05:23:57,0.0,-90.00,0.0,-0.5,0.0,0.0,-90.80,3,5
9228031,2018-01-11 06:58:42,2018-01-11 06:59:59,0.0,0.00,0.0,0.0,0.0,0.0,0.00,2,1
3447699,2018-08-29 10:09:01,2018-08-29 10:09:25,0.0,-2.50,0.0,-0.5,0.0,0.0,-3.30,3,1
9527560,2018-07-31 05:51:18,2018-07-31 05:52:23,0.0,0.00,0.0,0.0,0.0,0.0,0.00,2,1
2164320,2018-09-27 14:22:04,2018-09-27 14:22:14,0.0,-2.50,0.0,-0.5,0.0,0.0,-3.30,3,1
...,...,...,...,...,...,...,...,...,...,...,...
3589650,2018-12-24 08:23:18,2018-12-24 08:24:02,0.0,0.00,0.0,0.0,0.0,0.0,0.00,1,1
107823,2018-06-08 21:39:36,2018-06-08 21:41:08,0.0,-3.00,-0.5,-0.5,0.0,0.0,-4.30,4,1
6732692,2018-03-13 00:48:03,2018-03-14 00:46:49,0.0,-2.50,-0.5,-0.5,0.0,0.0,-3.80,3,1
1976874,2018-04-15 21:55:18,2018-04-15 21:56:39,0.0,0.00,0.0,0.0,7.0,0.0,7.30,1,5


We can see 35 lines where 'fare_amount' = 0 and 39 lines where 'fare_amount' < 0, and doing a more acurated analyse we can see that we have 'total_amount' with negatives and zero values, so lets check this

Noticible that 1995 lines have trip_distance = 0

In [39]:
trips[trips['total_amount'] <= 0].shape

(343, 17)

In [40]:
trips[trips['total_amount'] == 0].shape

(63, 17)

In [41]:
trips[trips['total_amount'] < 0].shape

(280, 17)

In [42]:
total_amount_invalid = trips[
    trips["total_amount"] <= 0
]

total_amount_invalid[[
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "payment_type",
    "rate_code"
]]

,pickup_datetime,dropoff_datetime,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount,payment_type,rate_code
1663721,2018-09-09 20:47:49,2018-09-09 20:48:02,0.01,-2.50,-0.5,-0.5,0.0,0.0,-3.80,3,1
2891353,2018-12-03 14:49:23,2018-12-03 14:49:41,0.04,-2.50,0.0,-0.5,0.0,0.0,-3.30,4,1
5493132,2018-12-29 05:22:29,2018-12-29 05:23:57,0.00,-90.00,0.0,-0.5,0.0,0.0,-90.80,3,5
439494,2018-03-12 15:18:54,2018-03-12 15:19:13,0.10,-2.50,0.0,-0.5,0.0,0.0,-3.30,3,1
6561509,2018-08-14 00:44:23,2018-08-15 00:34:28,0.67,-4.50,-0.5,-0.5,0.0,0.0,-5.80,4,1
...,...,...,...,...,...,...,...,...,...,...,...
9179008,2018-12-01 17:20:01,2018-12-01 17:20:01,0.00,0.00,0.0,0.0,0.0,0.0,0.00,2,5
8273062,2018-03-18 05:20:48,2018-03-18 05:22:49,0.27,-3.50,-0.5,-0.5,0.0,0.0,-4.80,3,1
1173710,2018-06-10 21:27:32,2018-06-10 21:31:00,0.47,-4.00,-0.5,-0.5,0.0,0.0,-5.30,3,1
6042796,2018-12-16 15:40:38,2018-12-16 15:44:39,0.00,-0.05,0.0,0.0,0.0,0.0,-0.35,4,5


We can see a correlation about total_amount beeing negative with other columns that compose the total_amount formula beeing negative too

## 5.3 Time series analysis

In [43]:
trips_temp = trips.copy()

trips_temp["pickup_datetime"] = pd.to_datetime(
    trips_temp["pickup_datetime"]
)

trips_temp["dropoff_datetime"] = pd.to_datetime(
    trips_temp["dropoff_datetime"]
)

trips_temp[["pickup_datetime", "dropoff_datetime"]].dtypes

pickup_datetime     datetime64[ns]
dropoff_datetime    datetime64[ns]
dtype: object

In [44]:
trips_temp["pickup_datetime"].min()

Timestamp('2009-01-01 00:48:08')

In [45]:
trips_temp["pickup_datetime"].max()

Timestamp('2018-12-31 23:45:24')

In [46]:
trips_temp["dropoff_datetime"].min()

Timestamp('2009-01-01 01:02:47')

In [47]:
trips_temp["dropoff_datetime"].max()

Timestamp('2019-01-01 00:17:51')

The sample covers the period defined by the minimum and maximum values observed in `pickup_datetime` and `dropoff_datetime`.

In [48]:
trips_temp["trip_duration"] = (
    trips_temp["dropoff_datetime"]
    - trips_temp["pickup_datetime"]
)

trips_temp["trip_duration_minutes"] = (
    trips_temp["trip_duration"].dt.total_seconds() / 60
)

trips_temp["trip_duration_minutes"].describe()

count    100000.000000
mean         36.166737
std          83.716733
min         -30.100000
25%          22.700000
50%          30.083333
75%          38.666667
max        1439.716667
Name: trip_duration_minutes, dtype: float64

We can see some errors because how a trip has -30 minutes? So now we will explore more in this

In [49]:
(trips_temp["trip_duration_minutes"] < 0).sum()

np.int64(1)

We have 2 trips that have a duration less than 0 minutes

In [50]:
(trips_temp["trip_duration_minutes"] == 0).sum()

np.int64(211)

We have 92 trips that have a duration equal 0 minutes

First lets see what is in the negative time data lines

In [51]:
negative_duration = trips_temp[trips_temp["trip_duration_minutes"] < 0]

negative_duration[
    [
        "pickup_datetime",
        "dropoff_datetime",
        "trip_duration_minutes",
        "trip_distance",
        "fare_amount",
        "pickup_location_id",
        "dropoff_location_id"
    ]
]

,pickup_datetime,dropoff_datetime,trip_duration_minutes,trip_distance,fare_amount,pickup_location_id,dropoff_location_id
3325941,2018-11-04 01:45:51,2018-11-04 01:15:45,-30.1,5.6,23.5,90,37


### Trip Duration Findings

Trip duration was calculated from the difference between `dropoff_datetime`
and `pickup_datetime`.

The sample contains:
- 2 trips with negative duration;
- 92 trips with zero duration.

Negative durations are considered temporally inconsistent because the
drop-off timestamp occurs before the pick-up timestamp. Since there is not
enough evidence to safely correct these timestamps, these records will be
treated as invalid and removed during the pipeline transformation stage.

Zero-duration trips require contextual analysis because a zero duration
does not necessarily imply that the entire trip record is invalid.

In [52]:
trips_temp["pickup_hour"] = trips_temp["pickup_datetime"].dt.hour

trips_by_hour = (
    trips_temp["pickup_hour"]
    .value_counts()
    .sort_index()
)

trips_by_hour

pickup_hour
0     3642
1     2141
2     1368
3     1139
4     1254
5     1441
6     2189
7     3205
8     4227
9     4454
10    4543
11    4785
12    5042
13    5346
14    5537
15    5693
16    5477
17    5493
18    5646
19    5357
20    5135
21    5760
22    5798
23    5328
Name: count, dtype: int64

### Analyse trips by hours of the day

In [53]:
fig1 = px.bar(
    x=trips_by_hour.index,
    y=trips_by_hour.values,
    labels={"x": "Hour of day", "y": "Number of trips"},
    title="Trips by pickup hour",
    text=trips_by_hour.values
)

fig1.show()

Pickup volume follows a clear daily cycle. The lowest activity occurs
between 2 AM and 5 AM (roughly 1,100-1,400 trips/hour), consistent with
overnight low demand. Volume ramps up steadily from 6 AM onward, reaching
a stable elevated plateau (~5,100-5,800 trips/hour) from around noon
through the evening. The two highest hours are 10 PM and 9 PM
(5,798 and 5,760 trips respectively), likely reflecting evening
leisure/nightlife demand.

In [54]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

trips_temp["pickup_day_of_week"] = trips_temp["pickup_datetime"].dt.day_name()

trips_by_day = trips_temp["pickup_day_of_week"].value_counts()
trips_by_day = trips_by_day.reindex(day_order)

trips_by_day

pickup_day_of_week
Monday       13388
Tuesday      14399
Wednesday    14543
Thursday     16462
Friday       16047
Saturday     12463
Sunday       12698
Name: count, dtype: int64

### Analyse trips by days of the week

In [55]:
fig = px.bar(
    x=trips_by_day.index,
    y=trips_by_day.values,
    labels={"x": "Day of the week", "y": "Number of trips"},
    title="Trips by days of the week",
    text=trips_by_day.values
)

fig.show()

Trip volume is clearly weekday-dominated, peaking on Thursday (16,462 trips)
and staying elevated Tuesday through Friday, with Friday close behind at
16,047. Both weekend days are lower and close to each other (Saturday:
12,463; Sunday: 12,698), roughly 23-25% below the Thursday peak. This
weekday-skewed pattern — combined with the evening peak seen in the hourly
breakdown (9-10 PM) — suggests the sample is dominated by commute/after-work
trips rather than weekend leisure travel.

### Analyse trips by month of the year

In [67]:
trips_temp["pickup_month_num"] = trips_temp["pickup_datetime"].dt.month
trips_temp["pickup_month_name"] = trips_temp["pickup_datetime"].dt.month_name()

trips_by_month = (
    trips_temp
    .groupby(["pickup_month_num", "pickup_month_name"])
    .size()
    .reset_index(name="count")
    .sort_values("pickup_month_num")
)

trips_by_month

,pickup_month_num,pickup_month_name,count
0,1,January,6989
1,2,February,7060
2,3,March,15707
3,4,April,8151
4,5,May,8692
5,6,June,8353
6,7,July,7087
7,8,August,6761
8,9,September,7479
9,10,October,8330


In [70]:
fig2 = px.bar(
    x=trips_by_month["pickup_month_name"],
    y=trips_by_month["count"],
    labels={"x": "Month", "y": "Number of trips"},
    title="Trips by month",
    text=trips_by_month["count"]
)

fig2.show()

## 5.4 Distributions

> **TODO — slot not started yet.**
>
> Histograms / boxplots for `trip_distance`, `fare_amount`, `trip_duration_minutes`, `tip_amount`, `total_amount`. Check skewness and whether a log scale is needed.

## 5.5 Outliers

> **Partially covered above.** Outliers already surfaced during 5.2 (Data quality) and 5.3 (Time series analysis):
>  - `trip_distance <= 0` → 1,899 rows (~1.9%)
>  - `pickup_location_id == dropoff_location_id` → 4,647 rows
>  - `trip_duration_minutes < 0` → 2 rows
>  - `trip_duration_minutes == 0` → 92 rows
>
> **TODO — remaining work for this slot:**
>
> - Consolidate the outlier findings above into one summary table
> - IQR / z-score based outlier detection for `fare_amount`, `trip_distance`, `trip_duration_minutes` (the checks above were rule-based, not statistical)
> - Boxplots for the numeric columns to visualize the outliers

## 5.6 Relationships between variables

> **TODO — slot not started yet.**
>
> Correlation matrix for numeric columns; scatter plots such as `trip_distance` × `fare_amount`, `trip_duration_minutes` × `trip_distance`; check if `payment_type` relates to `tip_amount`.

# 6. EDA - taxi_zone_geo

## 6.1 Structure

> **TODO — slot not started yet.**
>
> `shape`, `head`, `info`, dtypes for `zones` (beyond the initial `quick(zones)` check in section 3).

## 6.2 Quality

> **TODO — slot not started yet.**
>
> Nulls, duplicates, and value ranges for the zone geography fields.

## 6.3 Zone integrity

> **TODO — slot not started yet.**
>
> Check `location_id` is unique / a valid primary key, no orphan or duplicated zone ids.

# 7. Relationship: Trips × Zones

## 7.1 Pickup zones

> **TODO — slot not started yet.**
>
> Distribution of trips by `pickup_location_id`, top/bottom zones by volume.

## 7.2 Dropoff zones

> **TODO — slot not started yet.**
>
> Same analysis as 7.1, but for `dropoff_location_id`.

## 7.3 Unmatched records

> **TODO — slot not started yet.**
>
> Check whether every `pickup_location_id` / `dropoff_location_id` in `trips` exists in `zones` (anti-join); quantify unmatched rows.

# 8. Conclusions

> **TODO — slot not started yet.**
>
> Summarize the key findings from sections 5–7 into a short narrative once they are complete.

# 9. Pipeline requirements

> **Draft — based on findings so far, to be finalized after sections 5.4–8 are complete.**
>
> - Convert `pickup_datetime` and `dropoff_datetime` from object to datetime
> - Create an auxiliary lookup table for `payment_type` codes
> - Treat `rate_code` and `store_and_fwd_flag` as categorical
> - Remove exact duplicate rows (134 duplicates / 268 rows in the 100k sample)
> - Filter out rows with `trip_distance <= 0` (~1.9% of the 100k sample)
> - Remove rows with negative `trip_duration_minutes` (pickup_datetime > dropoff_datetime)
> - Decide how to handle rows with `trip_duration_minutes == 0` (92 rows) — pending investigation
> - Decide how to handle `pickup_location_id == dropoff_location_id` rows (4,647 rows) — pending investigation